# Two-Stage Amplifier Design

## Imports and Paths

In [1]:
import os
import sys
import subprocess

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sympy as sp

from tabulate import tabulate

sys.path.append(os.path.abspath(".."))
from pyIC.ic_utils import IcUtils as ic
from pyIC.xschem_utils import XschemUtils as xschem

analog_explorer_path = '/home/andreasp/university/projects/analog-explorer'

## AC Response Requirements

The 2nd stage pole is placed at the GBW frequency to achieve a first-order responde. The pole is given by:

$f_{p2} = \frac{g_{m9}}{2 \pi C_L}$

In [6]:
# DC gain of 120 dB
Av_db   = 120
Av      = ic.db2gain(Av_db)

# Dominant pole at 10 Hz
fp1     = 10
GBW     = fp1 * Av

print(f'GBW: {ic.eng_format(GBW)}')

GBW: 10.000e6


## 2nd Stage Sizing

In [34]:
## 2nd stage nmos operating point

gmid_9  = 3
gm_9    = GBW * 2 * np.pi * (500e-15)
id_9    = gm_9 / gmid_9

m9_op = ic.getop(
    analog_explorer_path,
    model='hv_nmos',
    length=5,
    vds=1.6,
    gmid=gmid_9,
    id=id_9*1e9
)

ic.printop(m9_op, title="M9")

╭──────┬────────────╮
│ M9   │            │
├──────┼────────────┤
│ gmid │    2.983e0 │
│ vgs  │    1.290e0 │
│ gmro │  338.900e0 │
│ ft   │  141.441e6 │
│ l    │    5.000e0 │
│ w    │    2.356e3 │
│ gm   │  31.239e-6 │
│ ro   │   10.849e6 │
│ id   │  10.472e-6 │
│ cgg  │ 35.151e-15 │
╰──────┴────────────╯


In [9]:
## 2nd stage pmos operating point

m10_op = ic.getop(
    analog_explorer_path,
    model='hv_pmos',
    length=5,
    vds=1.6,
    gmid=3,
    id=id_9*1e9
)

ic.printop(m10_op, title="M10")

╭───────┬─────────────╮
│ M10   │             │
├───────┼─────────────┤
│ gmid  │     3.001e0 │
│ vgs   │     1.310e0 │
│ gmro  │     1.403e3 │
│ ft    │    34.578e6 │
│ l     │     5.000e0 │
│ w     │     8.456e3 │
│ gm    │   31.425e-6 │
│ ro    │    44.654e6 │
│ id    │   10.472e-6 │
│ cgg   │ 144.642e-15 │
╰───────┴─────────────╯


## 1st Stage Requirements

The 1st stage gain requirement is determined from the gain of the 2nd stage. Assuming a compensation capacitor of 150fF, the required output impedance of the first stage is determined by placing the dominant pole at the 3dB point.

$f_{p1} = \frac{1}{2 \pi R_{out} C_C (1 + A_{V,2})}$

In [11]:
Av_2 = m9_op['gm'] * ic.parallel([m10_op['ro'], m9_op['ro']])
Av_2_db = ic.gain2db(Av_2)
print(f'Second stage gain: {Av_2_db} dB')
Av_1_db = Av_db - Av_2_db
Av_1 = ic.db2gain(Av_1_db)
print(f'First stage gain: {Av_1_db} dB')

Cc = 150e-15
R_out1 = (1 / (2 * np.pi * Cc * (1 + Av_2))) / fp1
print(f'First stage Rout: {ic.eng_format(R_out1)}')

gm_1 = Av_1 / R_out1
print(f'First stage GM: {ic.eng_format(gm_1)}')

Second stage gain: 48.712358759539015 dB
First stage gain: 71.28764124046099 dB
First stage Rout: 387.723e6
First stage GM: 9.459e-6


## 1st Stage Sizing

In [14]:
## input pmos

gmid_1 = 15
id_1 = gm_1 / gmid_1

m1_op = ic.getop(
    analog_explorer_path,
    model='hv_pmos',
    length=5,
    vds=0.8,
    gmid=gmid_1,
    id=id_1*1e9
)

ic.printop(m1_op)

╭──────┬─────────────╮
│ gmid │    14.773e0 │
│ vgs  │  710.000e-3 │
│ gmro │     3.999e3 │
│ ft   │     4.789e6 │
│ l    │     5.000e0 │
│ w    │    23.274e3 │
│ gm   │    9.316e-6 │
│ ro   │   429.199e6 │
│ id   │  630.623e-9 │
│ cgg  │ 309.610e-15 │
╰──────┴─────────────╯


In [21]:
## input pmos cascode

m3_op = ic.getop(
    analog_explorer_path,
    model='hv_pmos',
    length=1,
    vds=0.8,
    gmid=10,
    id=id_1*1e9
)

ic.printop(m3_op)

pmos_casc_rout = (1 + m3_op['gmro']) * m1_op['ro'] + m3_op['ro']
print(f'PMOS Rout: {ic.eng_format(pmos_casc_rout)}')

╭──────┬────────────╮
│ gmid │    9.896e0 │
│ vgs  │ 790.000e-3 │
│ gmro │  602.245e0 │
│ ft   │  257.528e6 │
│ l    │    1.000e0 │
│ w    │    1.310e3 │
│ gm   │   6.241e-6 │
│ ro   │   96.505e6 │
│ id   │ 630.623e-9 │
│ cgg  │  3.857e-15 │
╰──────┴────────────╯
PMOS Rout: 259.009e9


In [29]:
## input nmos cascode

m5_op = ic.getop(
    analog_explorer_path,
    model='hv_nmos',
    length=2,
    vds=0.8,
    gmid=7,
    id=id_1*1e9
)

ic.printop(m5_op)

nmos_casc_rout = (1 + m5_op['gmro']) * m5_op['ro'] + m5_op['ro']
print(f'NMOS Rout: {ic.eng_format(nmos_casc_rout)}')

╭──────┬────────────╮
│ gmid │    7.065e0 │
│ vgs  │ 850.000e-3 │
│ gmro │  110.412e0 │
│ ft   │  352.713e6 │
│ l    │    2.000e0 │
│ w    │  368.200e0 │
│ gm   │   4.456e-6 │
│ ro   │   24.780e6 │
│ id   │ 630.623e-9 │
│ cgg  │  2.011e-15 │
╰──────┴────────────╯
NMOS Rout: 2.786e9


In [26]:
rout_1 = ic.parallel([nmos_casc_rout, pmos_casc_rout])
Av_1_real = m1_op['gm'] * rout_1
Av_1_real = ic.gain2db(Av_1_real)

print(f'1st stage Rout: {ic.eng_format(rout_1)}')
print(f'1st stage gain: {Av_1_real}')

1st stage Rout: 894.179e6
1st stage gain: 78.41351900443412


In [ ]:
m1_fingers  = 10
m3_fingers  = 1
m5_fingers  = 1
m9_fingers  = 10
m10_fingers = 10

twostage_op = {
    'M1': { # input pmos
        'w':  m1_op['w']/m1_fingers,
        'l':  m1_op['l'],
        'ng': m1_fingers,
        'm':  1
    },
    'M2': { # input pmos
        'w':  m1_op['w']/m1_fingers,
        'l':  m1_op['l'],
        'ng': m1_fingers,
        'm':  1
    },
    'M3': { # input pmos cascode
        'w':  m3_op['w']/m3_fingers,
        'l':  m3_op['l'],
        'ng': m3_fingers,
        'm':  1
    },
    'M4': { # input pmos cascode
        'w':  m3_op['w']/m3_fingers,
        'l':  m3_op['l'],
        'ng': m3_fingers,
        'm':  1
    },
    'M5': { # upper nmos cascode mirror
        'w':  m5_op['w']/m5_fingers,
        'l':  m5_op['l'],
        'ng': m5_fingers,
        'm':  1
    },
    'M6': { # upper nmos cascode mirror
        'w':  m5_op['w']/m5_fingers,
        'l':  m5_op['l'],
        'ng': m5_fingers,
        'm':  1
    },
    'M7': { # lower nmos cascode mirror
        'w':  m5_op['w']/m5_fingers,
        'l':  m5_op['l'],
        'ng': m5_fingers,
        'm':  1
    },
    'M8': { # lower nmos cascode mirror
        'w':  m5_op['w']/m5_fingers,
        'l':  m5_op['l'],
        'ng': m5_fingers,
        'm':  1
    },
    'M9': { # 2nd stage nmos
        'w':  m9_op['w']/m9_fingers,
        'l':  m9_op['l'],
        'ng': m9_fingers,
        'm':  1
    },
    'M10': { # 2nd stage pmos
        'w':  m10_op['w']/m10_fingers,
        'l':  m10_op['l'],
        'ng': m10_fingers,
        'm':  1
    },
    'M11': { # input pmos tail current source
        'w':  m1_op['w']/m1_fingers,
        'l':  m1_op['l'],
        'ng': m1_fingers,
        'm':  1
    },
    'M12': { # pmos current mirror
        'w':  m1_op['w']/m1_fingers,
        'l':  m1_op['l'],
        'ng': m1_fingers,
        'm':  1
    },
}

path = '/home/andreasp/projects/diy-ic/schematic/two_stage.sch'
xschem.edit_device(path, twostage_op)

Succesfully updated device(s).
